In [16]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
import seaborn as sns
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
import warnings
warnings.filterwarnings('ignore')



In [17]:
df = pd.read_csv("processed_df.csv")
df

,Session_Summary,tokens,words_count,sentence_count
0,we started our lecture with a recap of previou...,"['we', 'started', 'our', 'lecture', 'with', 'a...",224,16
1,"in this session, we explored various feature e...","['in', 'this', 'session', 'we', 'explored', 'v...",271,16
2,population and sample were further discussed u...,"['population', 'and', 'sample', 'were', 'furth...",336,22
3,we first looked at all the summaries and obser...,"['we', 'first', 'looked', 'at', 'all', 'the', ...",219,13
4,midsem metrics for evaluation and also discuss...,"['midsem', 'metrics', 'for', 'evaluation', 'an...",13,1
...,...,...,...,...
662,in this lecture we have learnt about what is p...,"['in', 'this', 'lecture', 'we', 'have', 'learn...",326,15
663,"today, we continued our discussion on statisti...","['today', 'we', 'continued', 'our', 'discussio...",97,5
664,"at the beginning, sir explained what are the w...","['at', 'the', 'beginning', 'sir', 'explained',...",231,16
665,we studied about crisp-dm\n(cross industry sta...,"['we', 'studied', 'about', 'crispdm', 'cross',...",106,6


In [18]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=100,  # Limit features to reduce dimensionality
    stop_words='english',  # Remove common English stop words
    min_df=2,  # Ignore terms that appear in less than 2 documents
    max_df=0.9  # Ignore terms that appear in more than 90% of documents
)

# Fit and transform the summaries to TF-IDF vectors
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Session_Summary'])

# Normalize the TF-IDF vectors
tfidf_matrix_normalized = normalize(tfidf_matrix)
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14916 stored elements and shape (667, 100)>
  Coords	Values
  (0, 80)	0.38548517045958375
  (0, 37)	0.2364095087580472
  (0, 93)	0.3074541943474751
  (0, 5)	0.23565084597368097
  (0, 52)	0.2666639364459283
  (0, 40)	0.22470748274014815
  (0, 72)	0.11952430481976059
  (0, 27)	0.436035774947729
  (0, 38)	0.15605275110276373
  (0, 45)	0.3279111176974556
  (0, 90)	0.08045256990148915
  (0, 24)	0.12219782772376934
  (0, 49)	0.13312739324129266
  (0, 4)	0.124788137063901
  (0, 92)	0.3109306317417318
  (0, 67)	0.13710436334323176
  (0, 3)	0.11217769047334904
  (0, 50)	0.06927187489013241
  (1, 93)	0.032822284497703205
  (1, 52)	0.037956958568008695
  (1, 45)	0.09334977105879134
  (1, 90)	0.03435486893886593
  (1, 92)	0.09958023512156569
  (1, 76)	0.050153288198563815
  (1, 97)	0.05165228632415522
  :	:
  (665, 58)	0.2795562004850144
  (665, 79)	0.1294067182868434
  (665, 48)	0.6070296736516859
  (665, 70)	0.36559059495851415
  (666

In [19]:
X_dense = tfidf_matrix_normalized.toarray()

# Function to evaluate clustering
def evaluate_clustering(X, labels, model_name):
    """
    Evaluate clustering using multiple metrics
    """
    # Skip evaluation if only one cluster was found
    if len(np.unique(labels)) <= 1:
        print(f"{model_name} resulted in only one cluster, skipping evaluation")
        return None
    
    # Calculate evaluation metrics
    try:
        silhouette = silhouette_score(X, labels)
    except:
        silhouette = "N/A"
    
    try:
        db_score = davies_bouldin_score(X, labels)
    except:
        db_score = "N/A"
    
    try:
        ch_score = calinski_harabasz_score(X, labels)
    except:
        ch_score = "N/A"
    
    # Print results
    print(f"\n{model_name} Evaluation:")
    print(f"Number of clusters: {len(np.unique(labels))}")
    print(f"Silhouette Score: {silhouette}")
    print(f"Davies-Bouldin Index: {db_score}")
    print(f"Calinski-Harabasz Index: {ch_score}")
    
    # Return metrics as dictionary
    return {
        'model': model_name,
        'n_clusters': len(np.unique(labels)),
        'silhouette': silhouette if silhouette != "N/A" else None,
        'davies_bouldin': db_score if db_score != "N/A" else None,
        'calinski_harabasz': ch_score if ch_score != "N/A" else None
    }

In [20]:
def visualize_clusters(X, labels, title, method='pca'):
    """
    Visualize clusters using dimensionality reduction
    """
    # Skip visualization if only one cluster was found
    if len(np.unique(labels)) <= 1:
        print(f"Only one cluster found, skipping visualization for {title}")
        return
    
    plt.figure(figsize=(12, 8))
    
    # Apply dimensionality reduction
    if method == 'pca':
        reducer = PCA(n_components=2)
        X_reduced = reducer.fit_transform(X)
        method_name = 'PCA'
    else:  # t-SNE
        reducer = TSNE(n_components=2, random_state=42)
        X_reduced = reducer.fit_transform(X)
        method_name = 't-SNE'
    
    # Create scatter plot
    scatter = plt.scatter(
        X_reduced[:, 0], 
        X_reduced[:, 1], 
        c=labels, 
        cmap='viridis', 
        alpha=0.7
    )
    
    # Add legend
    legend1 = plt.legend(*scatter.legend_elements(),
                        title="Clusters")
    plt.gca().add_artist(legend1)
    
    plt.title(f'{title} ({method_name} Visualization)')
    plt.xlabel(f'{method_name} Component 1')
    plt.ylabel(f'{method_name} Component 2')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


In [21]:
def analyze_clusters(df, labels, vectorizer, kmeans_model=None):
    """
    Analyze the content of each cluster
    """
    # Add cluster labels to dataframe
    df_with_clusters = df.copy()
    df_with_clusters['Cluster'] = labels
    
    # Get number of clusters
    n_clusters = len(np.unique(labels))
    
    # Print cluster sizes
    print("\nCluster sizes:")
    cluster_sizes = df_with_clusters['Cluster'].value_counts().sort_index()
    for cluster, size in cluster_sizes.items():
        print(f"Cluster {cluster}: {size} summaries ({size/len(df_with_clusters)*100:.1f}%)")
    
    # If KMeans model is provided, print top terms per cluster
    if kmeans_model is not None:
        print("\nTop terms per cluster:")
        order_centroids = kmeans_model.cluster_centers_.argsort()[:, ::-1]
        terms = vectorizer.get_feature_names_out()
        
        for i in range(n_clusters):
            print(f"Cluster {i}:")
            # Print top 10 terms for each cluster
            top_terms = [terms[ind] for ind in order_centroids[i, :10]]
            print(", ".join(top_terms))
            print()
    

In [22]:
print("\nSample summaries from each cluster:")
for i in range(n_clusters):
    cluster_df = df_with_clusters[df_with_clusters['Cluster'] == i]
    if len(cluster_df) > 0:
        print(f"Cluster {i} samples:")
        # Get 2 random samples from this cluster
        cluster_samples = cluster_df['Session_Summary'].sample(min(2, len(cluster_df))).values
        for j, sample in enumerate(cluster_samples):
            print(f"Sample {j+1}: {sample[:200]}...")  # Print first 200 chars
        print()

#---------------------------------
# DETERMINE OPTIMAL NUMBER OF CLUSTERS
#---------------------------------
print("\n--- Finding Optimal Number of Clusters ---")

# Use Yellowbrick's KElbowVisualizer for multiple metrics
metrics = ['distortion', 'silhouette', 'calinski_harabasz']
plt.figure(figsize=(18, 5))

for i, metric in enumerate(metrics):
    plt.subplot(1, 3, i+1)
    visualizer = KElbowVisualizer(
        KMeans(random_state=42, n_init=10), 
        k=(2, 10), 
        metric=metric,
        timings=False
    )
    visualizer.fit(tfidf_matrix_normalized)
    visualizer.finalize()

plt.tight_layout()
plt.show()

# Silhouette analysis for a range of k values
plt.figure(figsize=(18, 15))
k_range = range(2, 6)
for i, k in enumerate(k_range):
    plt.subplot(2, 2, i+1)
    visualizer = SilhouetteVisualizer(
        KMeans(n_clusters=k, random_state=42, n_init=10),
        colors='yellowbrick'
    )
    visualizer.fit(tfidf_matrix_normalized)
    visualizer.finalize()

plt.tight_layout()
plt.show()

# Based on the visualizations, choose an optimal k
# For this example, let's assume k=4 is optimal
# You should adjust this based on your results
optimal_k = 4
print(f"\nSelected optimal number of clusters: {optimal_k}")

#---------------------------------
# APPLY MULTIPLE CLUSTERING ALGORITHMS
#---------------------------------
print("\n--- Applying Multiple Clustering Algorithms ---")

# Store evaluation results
evaluation_results = []

# 1. K-means clustering
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(tfidf_matrix_normalized)
kmeans_eval = evaluate_clustering(tfidf_matrix_normalized, kmeans_labels, "K-means")
if kmeans_eval:
    evaluation_results.append(kmeans_eval)
visualize_clusters(X_dense, kmeans_labels, "K-means Clustering", method='pca')
visualize_clusters(X_dense, kmeans_labels, "K-means Clustering", method='tsne')

# 2. Agglomerative clustering
agg_clustering = AgglomerativeClustering(n_clusters=optimal_k)
agg_labels = agg_clustering.fit_predict(X_dense)
agg_eval = evaluate_clustering(X_dense, agg_labels, "Agglomerative Clustering")
if agg_eval:
    evaluation_results.append(agg_eval)
visualize_clusters(X_dense, agg_labels, "Agglomerative Clustering", method='pca')

# 3. DBSCAN clustering
# DBSCAN requires tuning of eps parameter
# Let's try a range of eps values
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5]
best_silhouette = -1
best_eps = None
best_dbscan_labels = None

for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    dbscan_labels = dbscan.fit_predict(X_dense)
    
    # Skip if all points are noise (-1)
    if len(np.unique(dbscan_labels)) <= 1:
        continue
    
    # Calculate silhouette score
    try:
        sil_score = silhouette_score(X_dense, dbscan_labels)
        if sil_score > best_silhouette:
            best_silhouette = sil_score
            best_eps = eps
            best_dbscan_labels = dbscan_labels
    except:
        continue

if best_dbscan_labels is not None:
    print(f"\nBest DBSCAN eps: {best_eps}")
    dbscan_eval = evaluate_clustering(X_dense, best_dbscan_labels, "DBSCAN")
    if dbscan_eval:
        evaluation_results.append(dbscan_eval)
    visualize_clusters(X_dense, best_dbscan_labels, "DBSCAN Clustering", method='pca')
else:
    print("\nDBSCAN could not find meaningful clusters with the tried parameters")

#---------------------------------
# COMPARE CLUSTERING RESULTS
#---------------------------------
print("\n--- Comparing Clustering Algorithms ---")

# Create comparison table
if evaluation_results:
    eval_df = pd.DataFrame(evaluation_results)
    print(eval_df)
    
    # Visualize comparison
    plt.figure(figsize=(12, 6))
    
    # Plot silhouette scores
    plt.subplot(1, 2, 1)
    sns.barplot(x='model', y='silhouette', data=eval_df)
    plt.title('Silhouette Score by Model')
    plt.ylim(0, 1)
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Plot Davies-Bouldin scores
    plt.subplot(1, 2, 2)
    sns.barplot(x='model', y='davies_bouldin', data=eval_df)
    plt.title('Davies-Bouldin Index by Model')
    plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

#---------------------------------
# ANALYZE BEST CLUSTERING RESULT
#---------------------------------
print("\n--- Analyzing Best Clustering Result ---")

# Choose the best clustering based on silhouette score
if evaluation_results:
    best_model = max(evaluation_results, key=lambda x: x['silhouette'] if x['silhouette'] is not None else -float('inf'))
    print(f"Best clustering model: {best_model['model']} with silhouette score {best_model['silhouette']}")
    
    # Use K-means for detailed analysis since it provides centroids
    analyze_clusters(df, kmeans_labels, tfidf_vectorizer, kmeans)
else:
    print("No valid clustering results to analyze")

#---------------------------------
# CONSENSUS CLUSTERING
#---------------------------------
print("\n--- Consensus Clustering Analysis ---")

# Check if we have multiple valid clusterings to compare
valid_clusterings = []
if 'kmeans_labels' in locals() and len(np.unique(kmeans_labels)) > 1:
    valid_clusterings.append(('K-means', kmeans_labels))
if 'agg_labels' in locals() and len(np.unique(agg_labels)) > 1:
    valid_clusterings.append(('Agglomerative', agg_labels))
if 'best_dbscan_labels' in locals() and best_dbscan_labels is not None and len(np.unique(best_dbscan_labels)) > 1:
    valid_clusterings.append(('DBSCAN', best_dbscan_labels))

if len(valid_clusterings) >= 2:
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    
    print("\nClustering agreement scores:")
    for i in range(len(valid_clusterings)):
        for j in range(i+1, len(valid_clusterings)):
            model_i, labels_i = valid_clusterings[i]
            model_j, labels_j = valid_clusterings[j]
            
            ari = adjusted_rand_score(labels_i, labels_j)
            nmi = normalized_mutual_info_score(labels_i, labels_j)
            
            print(f"{model_i} vs {model_j}:")
            print(f"  Adjusted Rand Index: {ari:.4f}")
            print(f"  Normalized Mutual Information: {nmi:.4f}")
            
            agreement_level = ""
            if ari > 0.8:
                agreement_level = "Very strong agreement"
            elif ari > 0.6:
                agreement_level = "Strong agreement"
            elif ari > 0.4:
                agreement_level = "Moderate agreement"
            elif ari > 0.2:
                agreement_level = "Weak agreement"
            else:
                agreement_level = "Very weak agreement"
                
            print(f"  Interpretation: {agreement_level}")
            print()
else:
    print("Not enough valid clustering results to perform consensus analysis")

print("\n--- Clustering Analysis Complete ---")


Sample summaries from each cluster:


NameError: name 'n_clusters' is not defined

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# For elbow method without Yellowbrick
def plot_elbow_method(X, k_range):
    inertias = []
    silhouette_scores = []
    ch_scores = []
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)
        
        if k > 1:  # Silhouette and CH scores require at least 2 clusters
            labels = kmeans.labels_
            silhouette_scores.append(silhouette_score(X, labels))
            ch_scores.append(calinski_harabasz_score(X, labels))
        else:
            silhouette_scores.append(0)
            ch_scores.append(0)
    
    # Plot the metrics
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    
    # Inertia (distortion)
    ax[0].plot(k_range, inertias, 'bo-')
    ax[0].set_xlabel('Number of clusters')
    ax[0].set_ylabel('Inertia')
    ax[0].set_title('Elbow Method (Inertia)')
    ax[0].grid(True)
    
    # Silhouette score
    ax[1].plot(k_range[1:], silhouette_scores[1:], 'ro-')
    ax[1].set_xlabel('Number of clusters')
    ax[1].set_ylabel('Silhouette Score')
    ax[1].set_title('Silhouette Method')
    ax[1].grid(True)
    
    # Calinski-Harabasz score
    ax[2].plot(k_range[1:], ch_scores[1:], 'go-')
    ax[2].set_xlabel('Number of clusters')
    ax[2].set_ylabel('Calinski-Harabasz Score')
    ax[2].set_title('Calinski-Harabasz Method')
    ax[2].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return inertias, silhouette_scores, ch_scores

# For silhouette visualization without Yellowbrick
def plot_silhouette(X, labels, n_clusters):
    from sklearn.metrics import silhouette_samples
    
    # Compute the silhouette scores for each sample
    silhouette_vals = silhouette_samples(X, labels)
    
    plt.figure(figsize=(8, 6))
    y_ticks = []
    y_lower, y_upper = 0, 0
    
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to cluster i
        cluster_silhouette_vals = silhouette_vals[labels == i]
        cluster_silhouette_vals.sort()
        
        y_upper += len(cluster_silhouette_vals)
        
        plt.barh(range(y_lower, y_upper), cluster_silhouette_vals, height=1.0, 
                edgecolor='none', color=plt.cm.viridis(i / n_clusters))
        
        # Add the cluster label at the middle of the cluster
        y_ticks.append((y_lower + y_upper) / 2)
        y_lower += len(cluster_silhouette_vals)
    
    # The vertical red line shows the average silhouette score
    plt.axvline(x=silhouette_score(X, labels), color="red", linestyle="--")
    
    plt.yticks(y_ticks, range(n_clusters))
    plt.ylabel('Cluster')
    plt.xlabel('Silhouette coefficient')
    plt.title(f'Silhouette Plot for {n_clusters} Clusters')
    plt.tight_layout()
    plt.show()
